# **Week 4: Evaluation, Tuning & Pipelines**

## Day 1: Building a Three-Way Split (Train / Validation / Test)

### Objectives
1. Perform a 3-way split (**60% Train**, **20% Validation**, **20% Test**) on a dataset with a fixed `random_state`.
2. Train a baseline model on the Training Set and tune a hyperparameter using ONLY the Validation Set.
3. Perform a final, honest evaluation on the untouched Test Set exactly **once**.
4. Document why tuning directly on the Test Set leads to Data Leakage and misleading results.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Fix random seeds
SEED = 42
np.random.seed(SEED)

# Generate a synthetic classification dataset (simulating Week 3 dataset)
X_raw, y_raw = make_classification(
    n_samples=1000, 
    n_features=10, 
    n_informative=8, 
    n_redundant=2, 
    random_state=SEED
)

# Convert to DataFrame 
feature_names = [f"feature_{i+1}" for i in range(10)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = pd.Series(y_raw, name="target")

print(f"Dataset Shape: X = {X.shape}, y = {y.shape}")

Dataset Shape: X = (1000, 10), y = (1000,)


### Step 1: Creating a 60 / 20 / 20 Train / Validation / Test Split
To construct a three-way split using `Scikit-learn`'s `train_test_split`:
1. First, split the dataset into **Train + Validation (80%)** and **Test (20%)**.
2. Second, split the **Train + Validation** set into **Train (75% of 80% = 60%)** and **Validation (25% of 80% = 20%)**.

In [3]:
# Step 1.1: Split into Train+Val (80%) and Test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Step 1.2: Split Train+Val into Train (60% total) and Validation (20% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=SEED, stratify=y_train_val
)

# Step 1.3: Verify shapes and proportions
total_samples = len(X)
print(f"Total Samples : {total_samples}")
print(f"Train Set     : {X_train.shape[0]} samples ({X_train.shape[0]/total_samples:.0%})")
print(f"Val Set       : {X_val.shape[0]} samples ({X_val.shape[0]/total_samples:.0%})")
print(f"Test Set      : {X_test.shape[0]} samples ({X_test.shape[0]/total_samples:.0%})")

Total Samples : 1000
Train Set     : 600 samples (60%)
Val Set       : 200 samples (20%)
Test Set      : 200 samples (20%)


### Step 2: Model Training and Hyperparameter Tuning
We train a `DecisionTreeClassifier` on the **Training Set (`X_train`, `y_train`)** and evaluate different `max_depth` hyperparameter values on the **Validation Set (`X_val`, `y_val`)**.

*Note: The Test Set remains completely untouched during this phase.*

In [9]:
depths = [1, 2, 3, 5, 7, 10, 15, None]
best_depth = None
best_val_acc = 0.0
results = []

for depth in depths:
    # 1. Train model ONLY on training set
    clf = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    clf.fit(X_train, y_train)
    
    # 2. Evaluate ONLY on validation set
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    val_acc = accuracy_score(y_val, clf.predict(X_val))
    
    results.append({
        'max_depth': depth,
        'Train Accuracy': round(train_acc, 4),
        'Validation Accuracy': round(val_acc, 4)
    })
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_depth = depth
        
# Display results table
tuning_df = pd.DataFrame(results)
print("---Hyperparameter Tuning Results (Validation Set)---")
print(tuning_df.to_string(index=False))
print(f"\nSelected Best Hyperparameter: max_depth = {best_depth} (Val Accuracy: {best_val_acc:.4f})")
if depths.count(best_depth) > 0 and sum(1 for r in results if r['Validation Accuracy'] == best_val_acc) > 1:
    print("Note: In case of a tie in Validation Accuracy, the smaller max_depth is preferred to reduce model complexity.")

---Hyperparameter Tuning Results (Validation Set)---
 max_depth  Train Accuracy  Validation Accuracy
       1.0          0.6183                0.620
       2.0          0.6333                0.590
       3.0          0.7217                0.695
       5.0          0.8367                0.805
       7.0          0.8833                0.810
      10.0          0.9517                0.820
      15.0          1.0000                0.835
       NaN          1.0000                0.835

Selected Best Hyperparameter: max_depth = 15 (Val Accuracy: 0.8350)
Note: In case of a tie in Validation Accuracy, the smaller max_depth is preferred to reduce model complexity.


### Step 3: Final Model Evaluation on Unseen Test Set
Now that the final hyperparameter configuration (`max_depth = 15`) is chosen, we train the final model on `X_train` (or `X_train + X_val`) and evaluate it **exactly once** on the **Test Set (`X_test`, `y_test`)**.

In [6]:
#Combine Train and Validation sets for final training
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

# Train final model with the selected best_depth
final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED)
final_model.fit(X_train_full, y_train_full)

# Predict on test set exactly once
y_test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("="*50)
print(f"FINAL TEST SET ACCURACY: {test_accuracy:.4f}")
print("="*50)
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

FINAL TEST SET ACCURACY: 0.7950

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.80      0.79      0.79       100
           1       0.79      0.80      0.80       100

    accuracy                           0.80       200
   macro avg       0.80      0.80      0.79       200
weighted avg       0.80      0.80      0.79       200



### Step 4: Why Tuning Against the Test Set is Dangerous (Data Leakage)

#### 1. Optimization Bias & Test Set Contamination
If we tune hyperparameters directly using the **Test Set**:
- The **Test Set** loses its independence and stops being "unseen data".
- The hyperparameter selection process implicitly learns and optimizes for the specific random noise and quirks present in the test set.

#### 2. False Optimism (Overestimating Generalization)
- Tuning against the test set yields an overly optimistic test score that cannot be replicated on real-world production data.
- The test set score becomes biased downwards in error (upwards in accuracy) because the parameters were explicitly picked to maximize performance on that specific set.

#### 3. The Role of the Three-Way Split
- **Train Set (60%):** Learns model weights ($w, b$ or tree splits).
- **Validation Set (20%):** Used for selecting hyperparameters and model choices.
- **Test Set (20%):** Kept in a "vault" and used **only once** at the very end to estimate true, unbiased real-world performance.